In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print(torch.cuda.device_count())


x = torch.randn(3, 3, device="cuda")
y = x @ x
print(x)
print(y)



1
tensor([[-1.0247,  0.1211,  0.2004],
        [-0.7249, -1.5488, -0.9107],
        [-0.9341,  0.0832,  1.4232]], device='cuda:0')
tensor([[ 0.7750, -0.2949, -0.0304],
        [ 2.7162,  2.2353, -0.0309],
        [-0.4326, -0.1235,  1.7627]], device='cuda:0')


In [1]:
import haiku as hk
import jax
import jax.numpy as jnp
from nucleotide_transformer.pretrained import get_pretrained_model

# Get pretrained model
parameters, forward_fn, tokenizer, config = get_pretrained_model(
    model_name="1B_agro_nt",
    embeddings_layers_to_save=(12,),
    max_positions=32,
)
forward_fn = hk.transform(forward_fn)


/home/htamm/miniconda3/envs/agront/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded model's hyperparameters.
Downloaded model's weights...


In [2]:
import random

nucleotides = ['A', 'C', 'G', 'T']

# y = number of sequences per list (full_list_x_y)
y = 1000
per_nuc = y // len(nucleotides)  # 250

# all sequence lengths x you want (6, 12, ..., 120)
x_values = list(range(6, 6 * 20 + 1, 6))  # [6, 12, ..., 120]

for x in x_values:
    random.seed(x)
    mid = x // 2

    sequences = []
    used = set()  # to track global uniqueness for this x

    for nuc in nucleotides:
        nuc_seqs = set()
        while len(nuc_seqs) < per_nuc:
            # generate random sequence
            seq_list = [random.choice(nucleotides) for _ in range(x)]
            # enforce middle nucleotide
            seq_list[mid] = nuc
            seq = ''.join(seq_list)

            # ensure this sequence is globally unique for this x
            if seq not in used:
                used.add(seq)
                nuc_seqs.add(seq)

        sequences.extend(nuc_seqs)

    sequences = sorted(sequences)

    var_name = f"full_list_{x}_{y}"
    globals()[var_name] = sequences

# sanity checks
print(len(full_list_6_1000), full_list_6_1000[:5])
print(len(full_list_120_1000), full_list_120_1000[:5])


1000 ['AAAATT', 'AAACAT', 'AAACCC', 'AAACCT', 'AAACTA']
1000 ['AAAACGGTCCGATGCAATTACCATGATCGTCCTTTCTCTGTCTCCTGCATAATTCCGACGGGCAAGAGTAGAGTACGGCATGGTGTCGTATACGAATCGCTATCTTCATATTAGTAGTGA', 'AAAACTGTCACAGCTCGCTGACCACAACTACGGAATAGGTCCCCTATCATATGTCCGAATTCTGGGCAAACAGGGTACAAGAATACTTACTTGAAGGGGGAAGGTCTCAGATGTGAAGAC', 'AAAACTTTATTAAAGCTCAATTGCACGCGCACTCGTGAGAATTTGAGCCAATTTAAGCACTACCATCTACCCACGCTTGTCGTCCCGATCCTCATTGGCATCCCCTACAGAAACCCACCT', 'AAACACCGGGGAGTTTCAAAACCGAAGCTGGGTAAGTTAGTGAGAAATATCGTGTGAAGCTGACTTAAAAGCTCGGTATGGGACCAACTGATACAAGGCGCGGTTAATAAAACCTATAGC', 'AAACAGGGCTCGGAAAGCACCATCGACAACTGGCTCAAATCCCCAGCGCGTGTCACAAGGCTAGAAGCGATGTAATTGGGGCAAATATTATGATAAGCAACCGACGGTTATTACCATCGG']


In [ ]:
import os
import torch

# Select GPU *before* importing JAX in a fresh kernel ideally
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import jax
import jax.numpy as jnp



print("Number of visible CUDA devices (PyTorch):", torch.cuda.device_count())

# ---- parameters for the lists you already created ----
y = 1000  # number of sequences per list: full_list_x_y
x_values = list(range(6, 6 * 20 + 1, 6))  # [6, 12, ..., 120]

# Optionally collect everything in a dict too
embeddings_by_length = {}

# One PRNG key, then split per x (good JAX style)
base_key = jax.random.PRNGKey(0)

for idx, x in enumerate(x_values):
    # Name of the sequence list created earlier
    seq_var_name = f"full_list_{x}_{y}"

    # Get the sequences from globals()
    sequences = globals()[seq_var_name]
    assert len(sequences) == y, f"Expected {y} sequences for length {x}, got {len(sequences)}"

    # Tokenize
    batch = tokenizer.batch_tokenize(sequences)
    token_ids = [b[1] for b in batch]       # b is typically (original_str, ids)
    tokens = jnp.asarray(token_ids, dtype=jnp.int32)  # shape: (batch, seq_len + specials)

    # Make a new random key for this batch
    key = jax.random.fold_in(base_key, idx)

    # Forward pass
    outs = forward_fn.apply(parameters, key, tokens)

    # Pick the layer you want – you wrote "layer 20", but your key was "embeddings_12"
    # Adjust this to the correct one for your model, e.g. "embeddings_20" if available.
    embeddings = outs["embeddings_12"]  # shape: (batch, seq_len_with_specials, hidden_dim)

    print(f"x={x}: embeddings_12.shape =", embeddings.shape)

    # Save to a variable name like embeddings_6_1000, embeddings_12_1000, ...
    emb_var_name = f"embeddings_{x}_{y}"
    globals()[emb_var_name] = embeddings

    # Also keep in a dict if you like indexed access
    embeddings_by_length[(x, y)] = embeddings


W1202 12:42:29.282712 1350657 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W1202 12:42:29.289333 1329139 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


(10, 32, 1500)


In [ ]:
# immer letztes layer + logits

import os
import torch

# GPU auswählen (am besten VOR dem ersten JAX-Import im frischen Kernel setzen)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import jax
import jax.numpy as jnp



print("Number of visible CUDA devices (PyTorch):", torch.cuda.device_count())

# --- Parameter wie vorher ---
y = 1000  # Anzahl Sequenzen pro Liste: full_list_x_y
x_values = list(range(6, 6 * 20 + 1, 6))  # [6, 12, ..., 120]

base_key = jax.random.PRNGKey(0)

# Optionales Dict für übersichtliche Sammlung
embeddings_last_by_length = {}
logits_by_length = {}

for idx, x in enumerate(x_values):
    # Name der Sequenzliste aus der vorherigen Zelle
    seq_var_name = f"full_list_{x}_{y}"
    sequences = globals()[seq_var_name]
    assert len(sequences) == y, f"Expected {y} sequences for length {x}, got {len(sequences)}"

    # Tokenisieren
    batch = tokenizer.batch_tokenize(sequences)
    token_ids = [b[1] for b in batch]              # b = (original_string, ids)
    tokens = jnp.asarray(token_ids, dtype=jnp.int32)

    # PRNG-Key für diesen Batch
    key = jax.random.fold_in(base_key, idx)

    # Forward-Pass
    outs = forward_fn.apply(parameters, key, tokens)

    # ---- 1. Embeddings aus dem letzten Layer holen ----
    if "hidden_states" in outs:
        # HuggingFace-Style: letzte Hidden-State-Ebene
        last_hidden = outs["hidden_states"][-1]
    else:
        # Fallback: höchstes embeddings_X nehmen
        emb_keys = [k for k in outs.keys() if k.startswith("embeddings_")]
        if not emb_keys:
            raise KeyError(f"Keine 'hidden_states' oder 'embeddings_*' in outs: {outs.keys()}")

        # nach Layer-ID sortieren, z.B. "embeddings_0", "embeddings_12"
        emb_keys_sorted = sorted(
            emb_keys,
            key=lambda s: int(s.split("_")[1])
        )
        last_key = emb_keys_sorted[-1]
        last_hidden = outs[last_key]

    print(f"x={x}: last_hidden.shape = {last_hidden.shape}")

    # Variable für die letzten Layer-Embeddings
    emb_var_name = f"embeddings_last_{x}_{y}"
    globals()[emb_var_name] = last_hidden
    embeddings_last_by_length[(x, y)] = last_hidden

    # ---- 2. Logits holen und speichern ----
    logits = None
    if "logits" in outs:
        logits = outs["logits"]
    elif "lm_logits" in outs:
        logits = outs["lm_logits"]

    if logits is not None:
        print(f"x={x}: logits.shape = {logits.shape}")
        logits_var_name = f"logits_{x}_{y}"
        globals()[logits_var_name] = logits
        logits_by_length[(x, y)] = logits
    else:
        print(f"x={x}: keine Logits in outs gefunden. Verfügbare Keys: {outs.keys()}")


In [3]:
print(outs["embeddings_12"])

[[[ -0.37829465  -7.817052     1.2317715  ...   2.403119     0.26740313
    -5.193776  ]
  [  3.6248162  -11.280587     9.808967   ...  -0.63659656  -5.446619
    -2.5782893 ]
  [  2.073442    -7.654725     6.6665025  ...   1.7031863    2.5718224
    -2.1831853 ]
  ...
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]]

 [[ -0.9202825   -6.171276     2.5364666  ...   3.795408     2.1836684
    -3.2639008 ]
  [  4.171793   -10.664946    12.369383   ...   1.0581734   -2.5833874
    -2.0383883 ]
  [  0.91910446  -5.6572685   12.832234   ...   4.4644156    3.8622031
    -2.9300437 ]
  ...
  [ 14.549945     1.5648042   22.035736   ...  12.752829    10.3577795
     2.566535  ]
  [ 14.549945     1.5648042   22.035736   ...  12.752829    10.3577795
     2.566535  ]
  [ 14.549945     1.5

In [ ]:
#!/usr/bin/env python3
"""
Einfaches Skript, um AgroNT-Embeddings für FASTA-Sequenzen zu berechnen
und deren Ähnlichkeit (Cosine Similarity) zu vergleichen.

Konfigurationen stehen direkt unten im Skript.
"""

from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np

from nucleotide_transformer.pretrained import get_pretrained_model


# ==========================
# Konfiguration
# ==========================

FASTA_DIR = Path("/home/htamm/models/agront/nucleotide-transformer/notebooks/fasta_files")
MODEL_NAME = "1B_agro_nt"
LAYER = 12           # letzter Layer (wie im Beispiel)
RNG_SEED = 0         # für reproduzierbare Ergebnisse


# ==========================
# Hilfsfunktionen
# ==========================

def read_fasta(path):
    """Sehr einfacher FASTA-Reader, gibt Liste von Sequenzen zurück."""
    seqs = []
    current = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current:
                    seqs.append("".join(current).upper())
                    current = []
            else:
                current.append(line)
        if current:
            seqs.append("".join(current).upper())
    return seqs


def cosine_similarity_matrix(X):
    """Cosine-Similarity-Matrix für Vektoren in X (n, d)."""
    norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    Xn = X / norms
    return Xn @ Xn.T  # (n, n)


# ==========================
# Hauptlogik
# ==========================

def main():
    # 1) FASTA-Dateien einlesen
    if not FASTA_DIR.is_dir():
        raise SystemExit(f"FASTA-Verzeichnis existiert nicht: {FASTA_DIR}")

    fasta_files = sorted(FASTA_DIR.glob("*.fa*"))
    if not fasta_files:
        raise SystemExit(f"Keine FASTA-Dateien in {FASTA_DIR} gefunden.")

    all_seqs_per_file = {}
    all_seqs_flat = []

    print(f"Suche FASTA-Dateien in {FASTA_DIR} ...")
    for ff in fasta_files:
        seqs = read_fasta(ff)
        if not seqs:
            print(f"Warnung: {ff.name} enthält keine Sequenzen, überspringe.")
            continue
        all_seqs_per_file[ff.name] = seqs
        all_seqs_flat.extend(seqs)

    if not all_seqs_per_file:
        raise SystemExit("Es wurden zwar FASTA-Dateien gefunden, aber keine Sequenzen.")

    # 2) maximale Sequenzlänge bestimmen
    max_len = max(len(s) for s in all_seqs_flat)
    print(f"Maximale Sequenzlänge über alle Dateien: {max_len}")

    # 3) Modell laden
    print(f"Lade Modell '{MODEL_NAME}' mit max_positions={max_len} ...")
    params, forward_fn, tokenizer, config = get_pretrained_model(
        model_name=MODEL_NAME,
        embeddings_layers_to_save=(LAYER,),
        max_positions=max_len,
    )
    forward_fn = hk.transform(forward_fn)

    # fixer Random Key
    rng = jax.random.PRNGKey(RNG_SEED)

    # 4) pro Datei Embeddings berechnen und Ähnlichkeit ausgeben
    for fname, seqs in all_seqs_per_file.items():
        print("\n" + "=" * 80)
        print(f"Datei: {fname}")
        print(f"  Anzahl Sequenzen: {len(seqs)}")
        print(f"  Sequenzlängen: {[len(s) for s in seqs]}")

        # Tokenisierung
        token_ids = [b[1] for b in tokenizer.batch_tokenize(seqs)]
        tokens = jnp.asarray(token_ids, dtype=jnp.int32)  # (batch, seq_len)

        # Inferenz
        outs = forward_fn.apply(params, rng, tokens)
        key = f"embeddings_{LAYER}"
        if key not in outs:
            raise KeyError(f"{key} nicht im Model-Output. Keys: {list(outs.keys())}")

        emb = np.array(outs[key])  # (n_seq, seq_len, hidden_dim)
        n_seq, seq_len, hidden_dim = emb.shape
        print(f"  Embeddings-Shape: (n_seq={n_seq}, seq_len={seq_len}, hidden_dim={hidden_dim})")

        # 4a) Mean-Pooling über die Sequenz
        mean_emb = emb.mean(axis=1)  # (n_seq, hidden_dim)

        # 4b) Embedding an der mittleren Position (SNP in der Mitte)
        mid_idx = seq_len // 2
        mid_emb = emb[:, mid_idx, :]  # (n_seq, hidden_dim)

        # 4c) Cosine-Similarity-Matrizen
        cos_mean = cosine_similarity_matrix(mean_emb)
        cos_mid = cosine_similarity_matrix(mid_emb)

        # Ausgabe (gerundet)
        print("  Cosine-Similarity (Mean-pooled):")
        print(np.round(cos_mean, 3))

        print("  Cosine-Similarity (Middle-Position):")
        print(np.round(cos_mid, 3))

        # Beispiel: erste zwei Sequenzen (falls vorhanden)
        if n_seq >= 2:
            def cos(u, v):
                return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12))

            print("  Beispiel (Seq 0 vs. Seq 1):")
            print(f"    Cos (Mean-pooled):      {cos(mean_emb[0], mean_emb[1]):.4f}")
            print(f"    Cos (Middle-Position):  {cos(mid_emb[0], mid_emb[1])::.4f}")

    print("\nFertig.")


if __name__ == "__main__":
    main()
